# 1. Project Information

**Project Name:** FRM_Week1_Fluid_Replacement Modeling (FRM)_with Gassmann's Equation

**Description:**
This notebook performs Fluid Replacement Modeling (FRM) (pemodelan substitusi fluida) on well-log data using Gassmann's equation. FRM predicts how the elastic response of a rock (Vp, Vs, density) would change if its pore fluid were replaced by another fluid, without needing a new well to be drilled. This is one of the core quantitative interpretation (QI) workflows in exploration geophysics.

**Objectives:**
- Build a Litho-Fluid Class (LFC) (kelas litologi-fluida) log from petrophysical logs.
- Estimate mineral and in-situ fluid elastic properties using Voigt-Reuss-Hill (VRH) averaging.
- Apply Gassmann fluid substitution to simulate brine, oil, and gas saturation scenarios.
- Validate the substitution results with quality-control (QC) checks.
- Visualize the results in log-track and crossplot form, following the same depth window and axis convention used in the reference study.

**Workflow:**
1. Load well-log data (`qsiwell2.csv`).
2. Classify Litho-Fluid Class (LFC).
3. Estimate mineral bulk modulus (VRH).
4. Estimate in-situ fluid density and bulk modulus.
5. Apply Gassmann substitution for brine, oil, and gas scenarios.
6. Compute acoustic impedance (Ip) and Vp/Vs ratio for every scenario.
7. Run data validation and QC checks.
8. Generate diagnostic visualizations.
9. Export the final results table.

**Dataset:** `qsiwell2.csv`, well 2 from the Quantitative Seismic Interpretation project dataset (Avseth, Mukerji & Mavko, 2005), as adapted by Amato del Monte (2015).



# 2. Import Libraries


In [1]:
# Standard library
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

# Third-party
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
%matplotlib inline


# 3. Configuration

All tunable parameters live here. Depth cutoffs, axis limits, and mineral/fluid constants are defined once and reused everywhere else in the notebook, so nothing below this section contains a hard-coded number.


In [2]:
PROJECT_NAME = "FRM_Week1_Gassmann_Fluid_Substitution"

# ----------------------- USER SETTINGS -----------------------

# Input CSV file (place it in the same folder as this notebook/script)
# Diasumsikan csv-nya ada di folder yang sama dengan notebook ini.
# Input dataset
DATA_PATH = Path("../data/qsiwell2.csv")

# Output directories
OUTPUT_PATH = Path("../outputs")
FIGURE_PATH = OUTPUT_PATH / "figures"

# Create output folders automatically
OUTPUT_PATH.mkdir(exist_ok=True)
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

# Analysis interval (set to None to use the entire well)
ANALYSIS_ZTOP = 2100.0      # None
ANALYSIS_ZBOT = 2400.0      # None

# Reservoir interval for zoomed plots
RESERVOIR_ZTOP = 2150.0
RESERVOIR_ZBOT = 2200.0

# Crossplot axis
IP_XLIM: Tuple[float, float] = (3000.0, 9000.0)
VPVS_YLIM: Tuple[float, float] = (1.5, 3.0)

# --------------------- INTERNAL SETTINGS ---------------------

DPI = 150
TICK_NBINS = 4

REQUIRED_COLUMNS: List[str] = [
    "DEPTH",
    "VP",
    "VS",
    "RHO",
    "VSH",
    "SWE",
    "PHIE",
]

# Litho-fluid classification thresholds
VSH_CUTOFF = 0.20
SW_CUTOFF = 0.90

In [3]:
@dataclass(frozen=True)
class MineralProperties:
    """Elastic constants of the mineral end members (quartz, clay)."""

    quartz_density_g_cc: float = 2.65
    quartz_bulk_modulus_gpa: float = 37.0
    quartz_shear_modulus_gpa: float = 44.0
    clay_density_g_cc: float = 2.81
    clay_bulk_modulus_gpa: float = 15.0
    clay_shear_modulus_gpa: float = 5.0


@dataclass(frozen=True)
class FluidProperties:
    """Elastic constants of the pore fluids (brine, oil, gas)."""

    brine_density_g_cc: float = 1.09
    brine_bulk_modulus_gpa: float = 2.8
    oil_density_g_cc: float = 0.78
    oil_bulk_modulus_gpa: float = 0.94
    gas_density_g_cc: float = 0.25
    gas_bulk_modulus_gpa: float = 0.06


MINERALS = MineralProperties()
FLUIDS = FluidProperties()

FACIES_COLORS: List[str] = ["#B3B3B3", "blue", "green", "red", "#996633"]
FACIES_NAMES: Dict[int, str] = {0: "undef", 1: "brine", 2: "oil", 3: "gas", 4: "shale"}
FACIES_CMAP = mcolors.ListedColormap(FACIES_COLORS, "indexed")


# 4. Helper Functions

Generic utilities with no rock-physics logic in them: directory handling, column validation, and figure/data export helpers.


In [4]:
def ensure_directory(path: Path) -> Path:
    """Create a directory (including parents) if it does not exist yet.

    Parameters
    ----------
    path : Path
        Directory to create.

    Returns
    -------
    Path
        The same path, guaranteed to exist.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path


def validate_columns(df: pd.DataFrame, required_columns: Iterable[str]) -> None:
    """Raise a ValueError if any required column is missing from df.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.
    required_columns : Iterable[str]
        Column names that must be present.

    Raises
    ------
    ValueError
        If one or more required columns are missing.
    """
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}")


def save_figure(fig: plt.Figure, filename: str, output_dir: Path, dpi: int = DPI) -> Path:
    """Save a matplotlib figure to output_dir and return the saved path.

    Parameters
    ----------
    fig : plt.Figure
        Figure to save.
    filename : str
        Output file name (e.g. "plot.png").
    output_dir : Path
        Directory to save into (created if necessary).
    dpi : int
        Resolution in dots per inch.

    Returns
    -------
    Path
        Full path of the saved figure.
    """
    ensure_directory(output_dir)
    file_path = output_dir / filename
    fig.savefig(file_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return file_path


def save_dataframe(df: pd.DataFrame, filename: str, output_dir: Path) -> Path:
    """Save a DataFrame to output_dir as CSV and return the saved path.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to save.
    filename : str
        Output file name (e.g. "results.csv").
    output_dir : Path
        Directory to save into (created if necessary).

    Returns
    -------
    Path
        Full path of the saved file.
    """
    ensure_directory(output_dir)
    file_path = output_dir / filename
    df.to_csv(file_path, index=True)
    return file_path


# 5. Core Physics Equation Functions


These two functions are the "equation layer" of the notebook. They take plain arrays in and arrays out, they contain no DataFrame logic, and they are the *only* place where the actual rock-physics formulas live.

The design intent (as requested): if a different mixing rule or a different fluid-substitution model is needed in the future, only the function bodies in this section need to change. Every function in Section 6 (Processing Pipeline) is a thin DataFrame wrapper that calls into this section, so the wrappers never need to be touched when an equation changes.


## Equation 1 – Voigt-Reuss-Hill (VRH) Average

### Purpose
Estimate the effective elastic modulus (bulk modulus K and shear modulus mu) of a mixture of two or more phases (e.g., quartz and clay, or brine and oil), used for both the mineral matrix and the pore-fluid mixture.

### Input
Volume fraction, bulk modulus, and shear modulus of each phase.

### Equation
Voigt (upper) bound:
$$K_{Voigt} = \sum_i f_i K_i$$

Reuss (lower) bound:
$$K_{Reuss} = \left(\sum_i \frac{f_i}{K_i}\right)^{-1}$$

Hill's average:
$$K_{VRH} = \frac{K_{Voigt} + K_{Reuss}}{2}$$

The same three equations apply to the shear modulus mu. This formulation follows Mavko, Mukerji & Dvorkin (2009).

### Output
`k_voigt, k_reuss, mu_voigt, mu_reuss, k_vrh, mu_vrh`, each an array matching the input shape.


In [5]:
def voigt_reuss_hill(
    volume_fractions: np.ndarray, bulk_moduli: np.ndarray, shear_moduli: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Voigt-Reuss-Hill (VRH) average of elastic moduli for a phase mixture.

    Parameters
    ----------
    volume_fractions : np.ndarray
        Volume fraction of each phase, shape (n_phases,) or (n_samples, n_phases).
    bulk_moduli : np.ndarray
        Bulk modulus of each phase (GPa), same length as volume_fractions.
    shear_moduli : np.ndarray
        Shear modulus of each phase (GPa). Use zeros for fluids.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        k_voigt, k_reuss, mu_voigt, mu_reuss, k_vrh, mu_vrh.

    Notes
    -----
    Reference: Mavko, Mukerji & Dvorkin (2009), The Rock Physics Handbook.
    """
    fractions = np.array(volume_fractions).T
    k = np.resize(np.array(bulk_moduli), np.shape(fractions))
    mu = np.resize(np.array(shear_moduli), np.shape(fractions))
    axis = 0 if fractions.ndim == 1 else 1

    k_voigt = np.sum(fractions * k, axis=axis)
    k_reuss = 1.0 / np.sum(fractions / k, axis=axis)
    mu_voigt = np.sum(fractions * mu, axis=axis)
    with np.errstate(divide="ignore", invalid="ignore"):
        mu_reuss = 1.0 / np.sum(fractions / mu, axis=axis)
    k_vrh = (k_voigt + k_reuss) / 2.0
    mu_vrh = (mu_voigt + mu_reuss) / 2.0
    return k_voigt, k_reuss, mu_voigt, mu_reuss, k_vrh, mu_vrh


## Equation 2 – Gassmann Fluid Substitution

### Purpose
Predict Vp, Vs, and density (rho) of a rock saturated with fluid 2, given the measured elastic response of the same rock saturated with fluid 1. This is the central equation (persamaan inti) of the entire FRM workflow.

### Input
`vp1, vs1, rho1` (measured, fluid 1), `rho_fluid1, k_fluid1` (fluid 1 properties), `rho_fluid2, k_fluid2` (fluid 2 properties), `k_mineral` (mineral bulk modulus), `porosity`.

### Equation
Dry-frame bulk modulus (Gassmann, 1951, as formulated in Smith, Sondergeld & Rai, 2003):
$$K_{dry} = \frac{K_1 \left(\dfrac{\phi K_0}{K_{fl1}} + 1 - \phi\right) - K_0}{\dfrac{\phi K_0}{K_{fl1}} + \dfrac{K_1}{K_0} - 1 - \phi}$$

Saturated bulk modulus with the new fluid:
$$K_2 = K_{dry} + \frac{\left(1 - \dfrac{K_{dry}}{K_0}\right)^2}{\dfrac{\phi}{K_{fl2}} + \dfrac{1-\phi}{K_0} - \dfrac{K_{dry}}{K_0^2}}$$

Shear modulus is assumed fluid-independent, so $\mu_2 = \mu_1$ (Wang, 2001). Density and velocity follow directly:
$$\rho_2 = \rho_1 - \phi \rho_{fl1} + \phi \rho_{fl2} \qquad V_{p2} = \sqrt{\dfrac{K_2 + \frac{4}{3}\mu_2}{\rho_2}} \qquad V_{s2} = \sqrt{\dfrac{\mu_2}{\rho_2}}$$

### Output
`vp2, vs2, rho2, k2`.


In [6]:
def gassmann_fluid_substitution(
    vp1: np.ndarray,
    vs1: np.ndarray,
    rho1: np.ndarray,
    rho_fluid1: np.ndarray,
    k_fluid1: np.ndarray,
    rho_fluid2: np.ndarray,
    k_fluid2: np.ndarray,
    k_mineral: np.ndarray,
    porosity: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Gassmann (1951) fluid substitution equation.

    Parameters
    ----------
    vp1, vs1 : np.ndarray
        Measured P- and S-wave velocity saturated with fluid 1, m/s.
    rho1 : np.ndarray
        Measured bulk density saturated with fluid 1, g/cc.
    rho_fluid1, k_fluid1 : np.ndarray
        Density (g/cc) and bulk modulus (GPa) of fluid 1.
    rho_fluid2, k_fluid2 : np.ndarray
        Density (g/cc) and bulk modulus (GPa) of fluid 2.
    k_mineral : np.ndarray
        Mineral (grain) bulk modulus, GPa.
    porosity : np.ndarray
        Effective porosity, fraction.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        vp2 (m/s), vs2 (m/s), rho2 (g/cc), k2 (GPa).

    Notes
    -----
    Reference: Smith, Sondergeld & Rai (2003); Wang (2001).
    """
    vp1_km_s = vp1 / 1000.0
    vs1_km_s = vs1 / 1000.0

    rho2 = rho1 - porosity * rho_fluid1 + porosity * rho_fluid2
    mu1 = rho1 * vs1_km_s ** 2.0
    k1 = rho1 * vp1_km_s ** 2.0 - (4.0 / 3.0) * mu1

    k_dry = (k1 * ((porosity * k_mineral) / k_fluid1 + 1.0 - porosity) - k_mineral) / (
        (porosity * k_mineral) / k_fluid1 + (k1 / k_mineral) - 1.0 - porosity
    )
    k2 = k_dry + (1.0 - (k_dry / k_mineral)) ** 2 / (
        (porosity / k_fluid2) + ((1.0 - porosity) / k_mineral) - (k_dry / k_mineral ** 2)
    )
    mu2 = mu1  # Gassmann assumes shear modulus is independent of pore fluid.

    vp2 = np.sqrt((k2 + (4.0 / 3.0) * mu2) / rho2) * 1000.0
    vs2 = np.sqrt(mu2 / rho2) * 1000.0
    return vp2, vs2, rho2, k2


# 6. Load Input Data

## Purpose
Load the well-log CSV into a DataFrame indexed by depth (kedalaman), and fail fast with a clear error if the file or a required column is missing.

## Input
File path to `qsiwell2.csv`.

## Output
DataFrame indexed by `DEPTH`, containing at least `VP`, `VS`, `RHO`, `VSH`, `SWE`, `PHIE`.


In [7]:
def load_data(
    file_path: Path,
    ztop: float | None = ANALYSIS_ZTOP,
    zbot: float | None = ANALYSIS_ZBOT,
) -> pd.DataFrame:
    """Load the well-log CSV into a DataFrame indexed by depth, optionally
    restricted to the depth window where petrophysical logs are fully
    populated.

    Parameters
    ----------
    file_path : Path
        Path to the well-log CSV file (e.g. qsiwell2.csv).
    ztop, zbot : float or None
        Depth window to keep (inclusive). Pass None for either bound to
        skip filtering on that side, so ztop=None, zbot=None uses the
        entire well. Defaults to ANALYSIS_ZTOP / ANALYSIS_ZBOT (2100-2400 m),
        the interval where RHO, SWE, and PHIE contain no missing values
        in the reference dataset. Filtering happens before any downstream
        calculation, so LFC classification and Gassmann substitution never
        operate on NaN inputs.

    Returns
    -------
    pd.DataFrame
        Well-log data indexed by DEPTH, filtered to [ztop, zbot] if given.

    Raises
    ------
    FileNotFoundError
        If file_path does not exist.
    ValueError
        If required columns are missing, or if the filtered window
        contains zero rows.
    """
    if not Path(file_path).exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")
    df = pd.read_csv(file_path)
    validate_columns(df, REQUIRED_COLUMNS)
    df = df.set_index("DEPTH")

    if ztop is not None:
        df = df[df.index >= ztop]
    if zbot is not None:
        df = df[df.index <= zbot]

    if df.empty:
        raise ValueError(
            f"No data left after filtering to depth window [{ztop}, {zbot}]. "
            "Check ztop/zbot or the DEPTH column of the input file."
        )
    return df

# 7. Processing Pipeline

Every step below follows the same contract: **DataFrame in, equation, DataFrame out**. Each function receives a DataFrame, copies it (`df = df.copy()`), adds new column(s), and returns it. None of these functions mutate the caller's DataFrame in place, and none of them do any plotting or file I/O.


## Step 1 – Restrict Analysis Depth Window

### Purpose
Trim the log to the depth interval where the petrophysical logs (VSH, SWE, PHIE) (log petrofisika) are actually computed. Outside this window those curves may be blank or unreliable, so every downstream step (LFC classification, statistics, plots) must run only on this restricted interval. This mirrors the reference study, which restricts to `2100-2400 m` before doing anything else.

### Input
DataFrame indexed by DEPTH, plus `ztop`/`zbot` depth bounds.

### Equation
$$
DEPTH \in [z_{top}, z_{bot}]
$$

### Output
DataFrame containing only rows where `ztop <= DEPTH <= zbot`.


In [8]:
def restrict_depth_window(df: pd.DataFrame, ztop: float = ANALYSIS_ZTOP, zbot: float = ANALYSIS_ZBOT) -> pd.DataFrame:
    """Restrict the DataFrame to the depth interval where logs are valid.

    Parameters
    ----------
    df : pd.DataFrame
        Well-log DataFrame indexed by DEPTH.
    ztop : float
        Shallowest depth to keep (inclusive).
    zbot : float
        Deepest depth to keep (inclusive).

    Returns
    -------
    pd.DataFrame
        Copy of df restricted to [ztop, zbot].

    Raises
    ------
    ValueError
        If the resulting DataFrame is empty (e.g. ztop/zbot outside the
        well's logged depth range).
    """
    df = df.copy()
    df = df[(df.index >= ztop) & (df.index <= zbot)]
    if df.empty:
        raise ValueError(
            f"No data left after restricting to [{ztop}, {zbot}]. "
            "Check that DATA_PATH points to a well covering this interval."
        )
    return df


## Step 2 – Classify Litho-Fluid Class (LFC)

### Purpose
Separate the log into Litho-Fluid Classes (kelas litologi-fluida): 0=undef, 1=brine sand, 2=oil sand, 3=gas sand, 4=shale. The in-situ log never contains class 3 (gas sand); that class only appears after fluid substitution in Step 5.

### Input
DataFrame with `VSH`, `SWE`.

### Equation
$$
LFC =
\begin{cases}
1 & \text{if } V_{sh} \le V_{sh,cutoff} \text{ and } S_w \ge S_{w,cutoff} \quad (\text{brine sand}) \\
2 & \text{if } V_{sh} \le V_{sh,cutoff} \text{ and } S_w < S_{w,cutoff} \quad (\text{oil sand}) \\
4 & \text{if } V_{sh} > V_{sh,cutoff} \quad (\text{shale}) \\
0 & \text{otherwise (undefined)}
\end{cases}
$$

### Output
DataFrame with an added `LFC` column (int).


In [9]:
def classify_lithofluid_class(
    df: pd.DataFrame, vsh_cutoff: float = VSH_CUTOFF, sw_cutoff: float = SW_CUTOFF
) -> pd.DataFrame:
    """Classify each depth sample into a Litho-Fluid Class (LFC).

    Parameters
    ----------
    df : pd.DataFrame
        Must contain VSH and SWE columns.
    vsh_cutoff : float
        Shale volume fraction above which a sample is classified as shale.
    sw_cutoff : float
        Water saturation fraction above which a sand sample is brine sand.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added LFC column (int).
    """
    df = df.copy()
    is_sand = df["VSH"] <= vsh_cutoff
    is_brine_sand = is_sand & (df["SWE"] >= sw_cutoff)
    is_oil_sand = is_sand & (df["SWE"] < sw_cutoff)
    is_shale = df["VSH"] > vsh_cutoff

    lfc = np.zeros(len(df), dtype=int)
    lfc[is_brine_sand.values] = 1
    lfc[is_oil_sand.values] = 2
    lfc[is_shale.values] = 4
    df["LFC"] = lfc
    return df


## Step 3 – Calculate Mineral Bulk Modulus (K0)

### Purpose
Estimate the effective mineral (grain) bulk modulus from the shale/sand mixture using the VRH equation from Section 5.

### Input
DataFrame with `VSH`, `PHIE`.

### Equation
Shale and sand volumes are first normalized to exclude porosity, then passed to `voigt_reuss_hill`:
$$V_{sh,norm} = \frac{V_{sh}}{V_{sh} + V_{sand}} \qquad V_{sand,norm} = \frac{V_{sand}}{V_{sh}+V_{sand}} \qquad V_{sand} = 1 - V_{sh} - \phi$$

### Output
DataFrame with an added `K0` column (mineral bulk modulus, GPa).


In [10]:
def calculate_mineral_bulk_modulus(
    df: pd.DataFrame, minerals: MineralProperties = MINERALS
) -> pd.DataFrame:
    """Calculate the mineral (grain) bulk modulus K0 from VSH via VRH averaging.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain VSH and PHIE columns.
    minerals : MineralProperties
        Elastic constants of the mineral end members.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added K0 column (mineral bulk modulus, GPa).
    """
    df = df.copy()
    shale = df["VSH"].values
    # VSH + PHIE kadang > 1 di beberapa sample (inkonsistensi kecil di log,
    # 21 sample di qsiwell2 window 2100-2400 m, semuanya kebetulan shale
    # jadi tidak pernah ikut substitusi Gassmann, tapi tetap di-guard di
    # sini supaya K0 tidak pernah dihitung dari fraksi volume negatif).
    sand = np.clip(1.0 - shale - df["PHIE"].values, 1e-6, None)
    total = shale + sand
    shale_norm = shale / total
    sand_norm = sand / total

    _, _, _, _, k0, _ = voigt_reuss_hill(
        [shale_norm, sand_norm],
        [minerals.clay_bulk_modulus_gpa, minerals.quartz_bulk_modulus_gpa],
        [minerals.clay_shear_modulus_gpa, minerals.quartz_shear_modulus_gpa],
    )
    df["K0"] = k0
    return df


## Step 4 – Calculate In-Situ Fluid Density

### Purpose
Estimate the pore-fluid density (densitas fluida) using linear (arithmetic) volumetric mixing between brine and hydrocarbon.

### Input
DataFrame with `SWE`.

### Equation
$$\rho_{fluid} = S_w \, \rho_{brine} + (1-S_w)\, \rho_{oil}$$

### Output
DataFrame with an added `RHO_FLUID` column (g/cc).


In [11]:
def calculate_insitu_fluid_density(
    df: pd.DataFrame, fluids: FluidProperties = FLUIDS
) -> pd.DataFrame:
    """Calculate the in-situ pore-fluid density via linear volumetric mixing.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain SWE column.
    fluids : FluidProperties
        Elastic constants of the pore fluids. Hydrocarbon is assumed oil,
        matching the in-situ (undeveloped) fluid case.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added RHO_FLUID column (g/cc).
    """
    df = df.copy()
    water = df["SWE"].values
    hydrocarbon = 1.0 - water
    df["RHO_FLUID"] = water * fluids.brine_density_g_cc + hydrocarbon * fluids.oil_density_g_cc
    return df


## Step 5 – Calculate In-Situ Fluid Bulk Modulus

### Purpose
Estimate the pore-fluid bulk modulus (modulus ruah fluida) using the Reuss (harmonic) bound of the VRH equation, the standard mixing law for fluids since fluids carry no shear modulus.

### Input
DataFrame with `SWE`.

### Equation
$$K_{fluid} = \left(\frac{S_w}{K_{brine}} + \frac{1-S_w}{K_{oil}}\right)^{-1}$$

### Output
DataFrame with an added `K_FLUID` column (GPa).


In [12]:
def calculate_insitu_fluid_bulk_modulus(
    df: pd.DataFrame, fluids: FluidProperties = FLUIDS
) -> pd.DataFrame:
    """Calculate the in-situ pore-fluid bulk modulus via Reuss averaging.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain SWE column.
    fluids : FluidProperties
        Elastic constants of the pore fluids.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added K_FLUID column (GPa).
    """
    df = df.copy()
    water = df["SWE"].values
    hydrocarbon = 1.0 - water
    _, k_reuss, _, _, _, _ = voigt_reuss_hill(
        [water, hydrocarbon],
        [fluids.brine_bulk_modulus_gpa, fluids.oil_bulk_modulus_gpa],
        [0.0, 0.0],
    )
    df["K_FLUID"] = k_reuss
    return df


## Step 6 – Apply Gassmann Fluid Substitution

### Purpose
Apply the Gassmann equation (Section 5) to sand intervals (`LFC` 1 or 2) for one fluid scenario (brine, oil, or gas), producing a new set of elastic logs for that scenario. This function is called three times in the Main Pipeline, once per scenario, so the "swap the fluid" logic lives in one place.

### Input
DataFrame with `VP`, `VS`, `RHO`, `PHIE`, `K0`, `RHO_FLUID`, `K_FLUID`, `LFC`, plus the target fluid scenario name.

### Equation
Calls `gassmann_fluid_substitution` (Section 5) with fluid 2 set to the target scenario's density/bulk modulus, applied only where `LFC` is 1 or 2 (sand).

### Output
DataFrame with added `VP_<SFX>`, `VS_<SFX>`, `RHO_<SFX>` (elastic logs for the new fluid) and `LFC_<scenario initial>` (facies log for that scenario), where `<SFX>` is `FRMB`/`FRMO`/`FRMG` for brine/oil/gas.


In [13]:
SCENARIO_SUFFIXES: Dict[str, str] = {"BRINE": "FRMB", "OIL": "FRMO", "GAS": "FRMG"}
SCENARIO_LFC_CODE: Dict[str, int] = {"BRINE": 1, "OIL": 2, "GAS": 3}


def apply_gassmann_scenario(
    df: pd.DataFrame,
    scenario: str,
    fluids: FluidProperties = FLUIDS,
) -> pd.DataFrame:
    """Apply Gassmann fluid substitution to sand intervals for one scenario.

    Parameters
    ----------
    df : pd.DataFrame
        Must already contain VP, VS, RHO, PHIE, K0, RHO_FLUID, K_FLUID, LFC.
    scenario : str
        One of "BRINE", "OIL", "GAS".
    fluids : FluidProperties
        Elastic constants of the pore fluids.

    Returns
    -------
    pd.DataFrame
        Copy of df with added VP_<SFX>, VS_<SFX>, RHO_<SFX>, LFC_<initial>
        columns, where <SFX> is the scenario suffix (FRMB/FRMO/FRMG).

    Raises
    ------
    ValueError
        If scenario is not a recognized fluid scenario.
    """
    if scenario not in SCENARIO_SUFFIXES:
        raise ValueError(f"Unknown scenario '{scenario}', expected one of {list(SCENARIO_SUFFIXES)}")

    df = df.copy()
    suffix = SCENARIO_SUFFIXES[scenario]
    is_sand = df["LFC"].isin([1, 2]).values

    target_density = {
        "BRINE": fluids.brine_density_g_cc,
        "OIL": fluids.oil_density_g_cc,
        "GAS": fluids.gas_density_g_cc,
    }[scenario]
    target_bulk_modulus = {
        "BRINE": fluids.brine_bulk_modulus_gpa,
        "OIL": fluids.oil_bulk_modulus_gpa,
        "GAS": fluids.gas_bulk_modulus_gpa,
    }[scenario]

    vp2, vs2, rho2, _ = gassmann_fluid_substitution(
        df["VP"].values,
        df["VS"].values,
        df["RHO"].values,
        df["RHO_FLUID"].values,
        df["K_FLUID"].values,
        np.full(len(df), target_density),
        np.full(len(df), target_bulk_modulus),
        df["K0"].values,
        df["PHIE"].values,
    )

    vp_col, vs_col, rho_col = f"VP_{suffix}", f"VS_{suffix}", f"RHO_{suffix}"
    df[vp_col] = df["VP"]
    df[vs_col] = df["VS"]
    df[rho_col] = df["RHO"]
    df.loc[is_sand, vp_col] = vp2[is_sand]
    df.loc[is_sand, vs_col] = vs2[is_sand]
    df.loc[is_sand, rho_col] = rho2[is_sand]

    lfc_scenario = np.zeros(len(df), dtype=int)
    lfc_scenario[is_sand] = SCENARIO_LFC_CODE[scenario]
    lfc_scenario[df["LFC"].values == 4] = 4
    df[f"LFC_{scenario[0]}"] = lfc_scenario
    return df


## Step 7 – Calculate Acoustic Impedance (Ip)

### Purpose
Compute P-wave acoustic impedance (impedansi akustik gelombang-P) for every fluid scenario, the key attribute used in seismic amplitude interpretation.

### Input
DataFrame with `VP<suffix>`, `RHO<suffix>` for each requested suffix.

### Equation
$$I_p = V_p \cdot \rho$$

### Output
DataFrame with an added `IP<suffix>` column for each suffix.


In [14]:
def calculate_acoustic_impedance(
    df: pd.DataFrame, suffixes: Iterable[str] = ("", "_FRMB", "_FRMO", "_FRMG")
) -> pd.DataFrame:
    """Calculate P-wave acoustic impedance, Ip = Vp * rho, per fluid scenario.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain VP<suffix> and RHO<suffix> for every suffix requested.
    suffixes : Iterable[str]
        Column-name suffixes identifying each scenario ("" = in-situ).

    Returns
    -------
    pd.DataFrame
        Copy of df with an added IP<suffix> column for each suffix.
    """
    df = df.copy()
    for sfx in suffixes:
        df[f"IP{sfx}"] = df[f"VP{sfx}"] * df[f"RHO{sfx}"]
    return df


## Step 8 – Calculate Vp/Vs Ratio

### Purpose
Compute the Vp/Vs ratio (rasio Vp/Vs) for every fluid scenario, a diagnostic attribute for lithology and pore-fluid discrimination.

### Input
DataFrame with `VP<suffix>`, `VS<suffix>` for each requested suffix.

### Equation
$$\frac{V_p}{V_s}$$

### Output
DataFrame with an added `VPVS<suffix>` column for each suffix.


In [15]:
def calculate_vpvs_ratio(
    df: pd.DataFrame, suffixes: Iterable[str] = ("", "_FRMB", "_FRMO", "_FRMG")
) -> pd.DataFrame:
    """Calculate the Vp/Vs ratio for each fluid scenario.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain VP<suffix> and VS<suffix> for every suffix requested.
    suffixes : Iterable[str]
        Column-name suffixes identifying each scenario ("" = in-situ).

    Returns
    -------
    pd.DataFrame
        Copy of df with an added VPVS<suffix> column for each suffix.
    """
    df = df.copy()
    for sfx in suffixes:
        df[f"VPVS{sfx}"] = df[f"VP{sfx}"] / df[f"VS{sfx}"]
    return df


## Step 9 – Calculate Dry-Frame QC Ratio

### Purpose
Create a QC feature (fitur kontrol kualitas) that will be used in Section 8 to confirm the substitution was implemented correctly: in brine-sand samples, substituting brine back into brine should reproduce the original Vp almost exactly.

### Input
DataFrame with `VP`, `VP_FRMB`.

### Equation
$$QC_{ratio} = \frac{V_{p,FRMB}}{V_p}$$

### Output
DataFrame with an added `QC_RATIO` column.


In [16]:
def calculate_dry_frame_qc_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate a QC ratio comparing brine-substituted Vp to the original Vp.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain VP and VP_FRMB columns.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added QC_RATIO column.
    """
    df = df.copy()
    df["QC_RATIO"] = df["VP_FRMB"] / df["VP"]
    return df


# 8. Data Validation

Checks run after the processing pipeline: missing values, duplicated rows, and a physical sanity check (verifikasi kewajaran fisik) on the substitution result itself.


In [17]:
def check_missing_values(df: pd.DataFrame, columns: Iterable[str]) -> Dict[str, int]:
    """Return a dict of column -> count of NaN values for the given columns.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.
    columns : Iterable[str]
        Columns to check for missing values.

    Returns
    -------
    Dict[str, int]
        Mapping of column name to number of NaN values.
    """
    return {c: int(df[c].isna().sum()) for c in columns if c in df.columns}


def check_duplicated_rows(df: pd.DataFrame) -> int:
    """Return the number of fully duplicated rows in df.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.

    Returns
    -------
    int
        Number of duplicated rows.
    """
    return int(df.duplicated().sum())


def check_value_range(df: pd.DataFrame, column: str, min_value: float, max_value: float) -> int:
    """Return the number of out-of-range values for a given column.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to check.
    column : str
        Column to check.
    min_value, max_value : float
        Inclusive valid range.

    Returns
    -------
    int
        Number of samples outside [min_value, max_value].
    """
    out_of_range = (df[column] < min_value) | (df[column] > max_value)
    return int(out_of_range.sum())


def run_fluid_substitution_qc(df: pd.DataFrame, tolerance: float = 0.01, pure_brine_sw: float = 0.98) -> Dict[str, object]:
    """Run post-substitution quality-control checks.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain LFC and QC_RATIO columns.
    tolerance : float
        Maximum allowed deviation of QC_RATIO from 1.0 in brine-sand samples.

    Returns
    -------
    Dict[str, object]
        Summary of QC results (max deviation, n_failed, passed flag).

    Raises
    ------
    KeyError
        If LFC or QC_RATIO columns are missing.
    """

    brine_sand = df[(df["LFC"] == 1) & (df["SWE"] >= pure_brine_sw)]
    if brine_sand.empty:
        return {"max_deviation": 0.0, "n_failed": 0, "passed": True}
    deviation = (brine_sand["QC_RATIO"] - 1.0).abs()
    n_failed = int((deviation > tolerance).sum())
    return {
        "max_deviation": float(deviation.max()),
        "n_failed": n_failed,
        "passed": n_failed == 0,
    }


# 9. Visualization

Five diagnostic plots, matching the convention used in the reference study (Amato del Monte, 2015):

1. Ip histogram by in-situ facies.
2. Full well-log track (Vsh/Sw/phi, Ip, Vp/Vs, LFC).
3. In-situ Ip vs Vp/Vs crossplot.
4. Reservoir-zoomed comparison track (`2150-2200 m`), showing in-situ vs brine- vs gas-substituted Ip and Vp/Vs, colored by the **in-situ** LFC log (not the substituted one), with Ip and Vp/Vs axes locked to `(3000, 9000)` and `(1.5, 3.0)` and the x-axis tick count fixed to `TICK_NBINS` via `locator_params`, so the panel is directly comparable to the reference figure.
5. Four-panel Ip vs Vp/Vs crossplot: in-situ, brine, oil, and gas scenarios side by side.


In [18]:
def plot_facies_histogram(df: pd.DataFrame, output_dir: Path) -> Path:
    """Plot Ip histograms split by Litho-Fluid Class (in-situ).

    Parameters
    ----------
    df : pd.DataFrame
        Must contain IP and LFC columns.
    output_dir : Path
        Directory to save the figure into.

    Returns
    -------
    Path
        Path to the saved figure.
    """
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 2))
    for ax, lfc_code in zip(axes, [1, 2, 4]):
        subset = df.loc[df["LFC"] == lfc_code, "IP"]
        ax.hist(subset, bins=50, color="black")
        ax.set_title(FACIES_NAMES[lfc_code])
    return save_figure(fig, "facies_histogram.png", output_dir)


def plot_well_log_track(df: pd.DataFrame, output_dir: Path) -> Path:
    """Plot a 4-track summary of the in-situ well logs.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain VSH, SWE, PHIE, IP, VPVS, LFC.
    output_dir : Path
        Directory to save the figure into.

    Returns
    -------
    Path
        Path to the saved figure.
    """
    cluster = np.repeat(np.expand_dims(df["LFC"].values, 1), 100, 1)

    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(8, 6),)
    axes[0].plot(df["VSH"], df.index, "-g", label="Vsh")
    axes[0].plot(df["SWE"], df.index, "-b", label="Sw")
    axes[0].plot(df["PHIE"], df.index, "-k", label="phi")
    axes[0].legend(loc="lower right", fontsize=8)
    axes[1].plot(df["IP"], df.index, "-", color="0.5")
    axes[2].plot(df["VPVS"], df.index, "-", color="0.5")
    image = axes[3].imshow(cluster, interpolation="none", aspect="auto", cmap=FACIES_CMAP, vmin=0, vmax=4, origin="lower")
    cbar = fig.colorbar(image, ax=axes[3])
    cbar.set_label((15 * " ").join(["undef", "brine", "oil", "gas", "shale"]))
    cbar.set_ticks([])
    cbar.ax.tick_params(size=0)
    for ax, title in zip(axes, ["Vsh / Sw / phi", "Ip", "Vp/Vs", "LFC"]):
        ax.invert_yaxis()
        ax.set_title(title, fontsize=9)
        ax.grid()
    for ax in axes[1:]:
        ax.set_yticklabels([])
        ax.tick_params(axis='y', left=False)
    return save_figure(fig, "well_log_track.png", output_dir)


def plot_original_crossplot(df: pd.DataFrame, output_dir: Path) -> Path:
    """Plot the in-situ Ip vs Vp/Vs crossplot colored by LFC."""

    fig, ax = plt.subplots(figsize=(8, 6))

    scatter = ax.scatter(df["IP"], df["VPVS"], s=20, c=df["LFC"], marker="o", edgecolors="none", alpha=0.7, cmap=FACIES_CMAP, vmin=0, vmax=4,)
    ax.set_xlim(*IP_XLIM)
    ax.set_ylim(*VPVS_YLIM)
    ax.set_xlabel("Ip")
    ax.set_ylabel("Vp/Vs")
    ax.grid(True)
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label((15 * " ").join(["undef", "brine", "oil", "gas", "shale"]))
    cbar.set_ticks([])
    cbar.ax.tick_params(size=0)
    return save_figure(fig, "original_crossplot.png", output_dir)


def plot_frm_comparison_track(
    df: pd.DataFrame,
    output_dir: Path,
    ztop: float = RESERVOIR_ZTOP,
    zbot: float = RESERVOIR_ZBOT,
) -> Path:
    """Plot the reservoir-zoomed in-situ vs brine vs gas comparison track.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain IP, IP_FRMB, IP_FRMG, VPVS, VPVS_FRMB, VPVS_FRMG, LFC.
    output_dir : Path
        Directory to save the figure into.
    ztop, zbot : float
        Depth window to zoom into (defaults to the 2150-2200 m reservoir
        interval used throughout this study).

    Returns
    -------
    Path
        Path to the saved figure.
    """
    window = df[(df.index >= ztop) & (df.index <= zbot)]
    cluster = np.repeat(np.expand_dims(window["LFC"].values, 1), 100, 1)

    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(8, 6))
    axes[0].plot(window["VSH"], window.index, "-g", label="Vsh")
    axes[0].plot(window["SWE"], window.index, "-b", label="Sw")
    axes[0].plot(window["PHIE"], window.index, "-k", label="phi")
    axes[0].legend(loc="lower right", fontsize=8)

    axes[1].plot(window["IP_FRMG"], window.index, "-r", label="gas")
    axes[1].plot(window["IP_FRMB"], window.index, "-b", label="brine")
    axes[1].plot(window["IP"], window.index, "-", color="0.5", label="in-situ")
    axes[1].set_xlim(*IP_XLIM)
    axes[1].locator_params(axis="x", nbins=TICK_NBINS)

    axes[2].plot(window["VPVS_FRMG"], window.index, "-r")
    axes[2].plot(window["VPVS_FRMB"], window.index, "-b")
    axes[2].plot(window["VPVS"], window.index, "-", color="0.5")
    axes[2].set_xlim(*VPVS_YLIM)
    axes[2].locator_params(axis="x", nbins=TICK_NBINS)

    image = axes[3].imshow(cluster, interpolation="none", aspect="auto", cmap=FACIES_CMAP, vmin=0, vmax=4, origin="lower")
    cbar = fig.colorbar(image, ax=axes[3])
    cbar.set_label((15 * " ").join(["undef", "brine", "oil", "gas", "shale"]))
    cbar.set_ticks([])
    cbar.ax.tick_params(size=0)
    for ax, title in zip(axes, ["Vsh / Sw / phi", "Ip", "Vp/Vs", "LFC (in-situ)"]):
        ax.invert_yaxis()
        ax.set_title(title, fontsize=9)
        ax.grid()
    for ax in axes[1:]:
        ax.set_yticklabels([])
        ax.tick_params(axis='y', left=False)
    fig.suptitle(f"FRM comparison track, {ztop:.0f}-{zbot:.0f} m")
    return save_figure(fig, "frm_comparison_track.png", output_dir)


def plot_frm_crossplot(df: pd.DataFrame, output_dir: Path) -> Path:
    """Plot 4-panel Ip vs Vp/Vs crossplots: in-situ, brine, oil, gas.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain IP/VPVS/LFC columns for all four scenarios.
    output_dir : Path
        Directory to save the figure into.

    Returns
    -------
    Path
        Path to the saved figure.
    """
    fig, axes = plt.subplots(nrows=1, ncols=4, sharey=True, sharex=True, figsize=(16, 4))
    panels = [
        ("IP", "VPVS", "LFC", "in-situ"),
        ("IP_FRMB", "VPVS_FRMB", "LFC_B", "brine"),
        ("IP_FRMO", "VPVS_FRMO", "LFC_O", "oil"),
        ("IP_FRMG", "VPVS_FRMG", "LFC_G", "gas"),
    ]
    for ax, (ip_col, vpvs_col, lfc_col, title) in zip(axes, panels):
        ax.scatter(df[ip_col], df[vpvs_col], 20, df[lfc_col], marker="o", edgecolors="none", alpha=0.5, cmap=FACIES_CMAP, vmin=0, vmax=4)
        ax.set_title(title)
        ax.grid()
    axes[0].set_xlim(*IP_XLIM)
    axes[0].set_ylim(*VPVS_YLIM)
    axes[0].set_xlabel("Ip")
    axes[0].set_ylabel("Vp/Vs")
    return save_figure(fig, "frm_crossplot.png", output_dir)


# 10. Export Results

## Purpose
Persist the final processed DataFrame to disk (CSV) so it can be reused in later weeks (RPT, AVO forward modeling) without rerunning this notebook.

## Input
DataFrame, output directory.

## Output
Path to the saved CSV file.


In [19]:
def export_data(df: pd.DataFrame, output_dir: Path, filename: str = "frm_week1_results.csv") -> Path:
    """Export the final processed DataFrame to CSV.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to export.
    output_dir : Path
        Directory to export into.
    filename : str
        Output file name.

    Returns
    -------
    Path
        Path to the saved file.
    """
    return save_dataframe(df, filename, output_dir)


# 11. Main Pipeline

No processing logic below this point, only calls into the functions defined above. This is the only cell you should need to re-run end to end once `DATA_PATH` points at your actual `qsiwell2.csv`.


In [20]:
start_time = time.time()

df = load_data(DATA_PATH)
df = restrict_depth_window(df, ANALYSIS_ZTOP, ANALYSIS_ZBOT)

df = classify_lithofluid_class(df)
df = calculate_mineral_bulk_modulus(df)
df = calculate_insitu_fluid_density(df)
df = calculate_insitu_fluid_bulk_modulus(df)
df = calculate_acoustic_impedance(df, suffixes=("",))
df = calculate_vpvs_ratio(df, suffixes=("",))

for scenario in ("BRINE", "OIL", "GAS"):
    df = apply_gassmann_scenario(df, scenario)

df = calculate_acoustic_impedance(df, suffixes=("_FRMB", "_FRMO", "_FRMG"))
df = calculate_vpvs_ratio(df, suffixes=("_FRMB", "_FRMO", "_FRMG"))
df = calculate_dry_frame_qc_ratio(df)

missing_report = check_missing_values(df, REQUIRED_COLUMNS)
duplicate_count = check_duplicated_rows(df)
qc_report = run_fluid_substitution_qc(df)
n_phie_out_of_range = check_value_range(df, "PHIE", 0.0, 1.0)
n_swe_out_of_range = check_value_range(df, "SWE", 0.0, 1.0)
print("Missing values:", missing_report)
print("Duplicated rows:", duplicate_count)
print("PHIE out of [0,1]:", n_phie_out_of_range)
print("SWE out of [0,1]:", n_swe_out_of_range)
print("Fluid substitution QC:", qc_report)

ensure_directory(FIGURE_PATH)
plot_facies_histogram(df, FIGURE_PATH)
plot_well_log_track(df, FIGURE_PATH)
plot_original_crossplot(df, FIGURE_PATH)
plot_frm_comparison_track(df, FIGURE_PATH)
plot_frm_crossplot(df, FIGURE_PATH)

export_path = export_data(df, OUTPUT_PATH)

elapsed_time = time.time() - start_time


Missing values: {'VP': 0, 'VS': 0, 'RHO': 0, 'VSH': 0, 'SWE': 0, 'PHIE': 0}
Duplicated rows: 0
PHIE out of [0,1]: 0
SWE out of [0,1]: 0
Fluid substitution QC: {'max_deviation': 0.003150513624426754, 'n_failed': 0, 'passed': True}


# 12. Summary


In [21]:
print(f"Project           : {PROJECT_NAME}")
print(f"Rows              : {len(df)}")
print(f"Columns           : {len(df.columns)}")
print(f"Created features  : LFC, K0, RHO_FLUID, K_FLUID, VP/VS/RHO/IP/VPVS for BRINE/OIL/GAS, QC_RATIO")
print(f"Execution time    : {elapsed_time:.2f}s")
print(f"Exported to       : {export_path.resolve()}")
print(f"Figures saved to  : {FIGURE_PATH.resolve()}")


Project           : FRM_Week1_Gassmann_Fluid_Substitution
Rows              : 1968
Columns           : 38
Created features  : LFC, K0, RHO_FLUID, K_FLUID, VP/VS/RHO/IP/VPVS for BRINE/OIL/GAS, QC_RATIO
Execution time    : 1.83s
Exported to       : E:\KP\RFD\TimeLine\Week_1\FRM\outputs\frm_week1_results.csv
Figures saved to  : E:\KP\RFD\TimeLine\Week_1\FRM\outputs\figures


## References

- Amato del Monte, A. (2015). *seismic petrophysics* [Jupyter notebook]. The Leading Edge tutorial series, April and June 2015 issues.
- Avseth, P., Mukerji, T., & Mavko, G. (2005). *Quantitative Seismic Interpretation*. Cambridge University Press.
- Mavko, G., Mukerji, T., & Dvorkin, J. (2009). *The Rock Physics Handbook*. Cambridge University Press.
- Smith, T. M., Sondergeld, C. H., & Rai, C. S. (2003). Gassmann fluid substitutions: A tutorial. *Geophysics, 68*(2), 430-440.
- Wang, Z. (2001). Fundamentals of seismic rock physics. *Geophysics, 66*(2), 398-412.
